In [1]:
from agents import Order, Restaurant, User, Driver
from environment import Environment
from simulation import Simulation, generate_orders, get_order_rate
from policies.dispatch import GreedyPolicy, HungarianPolicy
from policies.repositioning import StaticPolicy
from routing import distrito_tec, get_closest_place_node_id
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib.cm as cm
import numpy as np
import osmnx as ox
import pandas as pd

sub_graph, routable_restaurants, residential_zones = distrito_tec()


In [2]:
routable_restaurants.head(5)

,element,id,geometry,amenity,brand,brand:wikidata,brand:wikipedia,cuisine,name,official_name,...,air_conditioning,website:menu,contact:instagram,phone,drive_through,building,fast_food,building:levels,height,nn
0,node,890657171,POINT (-100.29383 25.65435),cafe,Starbucks,Q37158,en:Starbucks,coffee_shop,Starbucks,Starbucks Coffee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.433046e+09
1,node,890657529,POINT (-100.29377 25.65441),restaurant,NaN,NaN,NaN,NaN,Costeñito,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.433046e+09
2,node,890657855,POINT (-100.29355 25.65472),restaurant,NaN,NaN,NaN,NaN,Wings Army,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.433046e+09
3,node,890666591,POINT (-100.28436 25.64771),restaurant,NaN,NaN,NaN,NaN,Manhattan,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.049531e+09
4,node,890667172,POINT (-100.28432 25.64783),bar,NaN,NaN,NaN,NaN,La Rambla,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.049531e+09


In [3]:
routable_restaurants.shape

(50, 40)

In [4]:
residential_zones.head(5)

,element,id,geometry,landuse,name,place,addr:city,addr:street,building:levels,residential,operator,type
0,relation,9463437,"POLYGON ((-100.29348 25.66187, -100.2936 25.66...",residential,La Florida,NaN,NaN,NaN,NaN,NaN,NaN,multipolygon
1,way,163034600,"POLYGON ((-100.27483 25.62377, -100.27426 25.6...",residential,Contry,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,way,163034602,"POLYGON ((-100.27739 25.64568, -100.27511 25.6...",residential,Contry Tesoro,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,way,163034603,"POLYGON ((-100.28282 25.64297, -100.28238 25.6...",residential,Contry Lux,neighbourhood,NaN,NaN,NaN,NaN,NaN,NaN
4,way,163034605,"POLYGON ((-100.28052 25.64492, -100.27925 25.6...",residential,Las Musas,neighbourhood,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
routable_restaurants.iloc[[0]]

,element,id,geometry,amenity,brand,brand:wikidata,brand:wikipedia,cuisine,name,official_name,...,air_conditioning,website:menu,contact:instagram,phone,drive_through,building,fast_food,building:levels,height,nn
0,node,890657171,POINT (-100.29383 25.65435),cafe,Starbucks,Q37158,en:Starbucks,coffee_shop,Starbucks,Starbucks Coffee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.433046e+09


In [6]:
#ORDER_RATE = 8    # orders/min off-peak  (~480/hr)
#ORDER_RATE = 25   # orders/min lunch peak (~1500/hr)  
#ORDER_RATE = 20   # orders/min dinner peak (~1200/hr)
#N_DRIVERS = 50    # undersupplied scenario
N_DRIVERS = 100   # balanced
#N_DRIVERS = 200   # oversupplied

In [7]:
# =========================================================
# SCENARIO SETUP
# To swap strategies, change DISPATCH_POLICY / REPOSITION_POLICY.
# Nothing else in this file needs to change.
# =========================================================

# ---- Parameters ----
N_USERS           = 10000
#N_DRIVERS         = 50
#ORDER_RATE        = 15    # orders per minute, unused if dynamic wrapper in place
DISPATCH_INTERVAL = 3    # seconds between batch dispatches
START_HOUR = 11
# ---- Swap policies here ----
DISPATCH_POLICY   = HungarianPolicy(pickup_radius=3000)  # or GreedyPolicy()
REPOSITION_POLICY = StaticPolicy()                       # or RLPolicy() when ready

# ---------------------------------------------------------
# 1. Build environment + simulation
# ---------------------------------------------------------

env = Environment(sub_graph)

sim = Simulation(
    env=env,
    dispatch_policy=DISPATCH_POLICY,
    repositioning_policy=REPOSITION_POLICY,
    step_size=10,
    dispatch_interval=DISPATCH_INTERVAL,
    start_hour=START_HOUR # Used for dynamic demand wrapper
)

# ---------------------------------------------------------
# 2. Add restaurants
#    Ratings sampled uniformly [3.0, 5.0] so the MNL
#    restaurant-choice model has something to work with.
# ---------------------------------------------------------

for i in range(len(routable_restaurants)):

    res_node = get_closest_place_node_id(
        routable_restaurants.iloc[[i]],
        sub_graph
    )

    sim.add_restaurant(Restaurant(
        restaurant_id=i,
        location=res_node,
        rating=round(random.uniform(3.0, 5.0), 1),
        capacity=10,
        avg_prep_time=780,
        service_radius=5000,
    ))

print(f"{len(sim.restaurants)} restaurants loaded")

# ---------------------------------------------------------
# 3. Sample residential nodes for users / drivers
# ---------------------------------------------------------

sampled_zones = residential_zones.sample(N_USERS, replace=True)

residential_nodes = [
    get_closest_place_node_id(sampled_zones.iloc[[i]], sub_graph)
    for i in range(N_USERS)
]

# ---------------------------------------------------------
# 4. Create users
# ---------------------------------------------------------

for i in range(N_USERS):
    sim.add_user(User(user_id=i, location=residential_nodes[i]))

print(f"{len(sim.users)} users created")

# ---------------------------------------------------------
# 5. Create drivers
# ---------------------------------------------------------

for i in range(N_DRIVERS):
    sim.add_driver(Driver(
        driver_id=i,
        location_node=random.choice(residential_nodes),
        speed=None,
    ))

print(f"{len(sim.drivers)} drivers created")
print(f"Dispatch  : {sim.dispatch_policy.__class__.__name__}")
print(f"Reposition: {sim.repositioning_policy.__class__.__name__}")
print("Simulation ready.")


50 restaurants loaded
10000 users created
100 drivers created
Dispatch  : HungarianPolicy
Reposition: StaticPolicy
Simulation ready.


In [8]:
SHOW_TRAILS = False

In [9]:
%matplotlib inline


# --- Map canvas ---
fig, ax = ox.plot_graph(
    sub_graph, bgcolor='k', edge_color='#333333',
    node_size=0, edge_linewidth=0.5, show=False, close=False
)

# --- Static restaurant markers ---
restaurant_lons = [sub_graph.nodes[r.location]['x'] for r in sim.restaurants.values()]
restaurant_lats = [sub_graph.nodes[r.location]['y'] for r in sim.restaurants.values()]

ax.scatter(
    restaurant_lons, restaurant_lats,
    s=120, c='red', marker='^',
    edgecolors='white', linewidth=1.2,
    zorder=12, label='Restaurants'
)

# --- Driver visual elements ---
drivers_to_watch = list(sim.drivers.values())
if not drivers_to_watch:
    raise ValueError("No drivers found — check setup cell.")

colors = cm.rainbow(np.linspace(0, 1, len(drivers_to_watch)))
dots   = ax.scatter([], [], c=[], s=60, zorder=10, edgecolors='white', linewidth=0.8)

if SHOW_TRAILS:
    lines = [ax.plot([], [], color=colors[i], lw=2.5, alpha=0.7, zorder=9)[0]
             for i in range(len(drivers_to_watch))]
else:
    lines = []

trail_history = [[] for _ in drivers_to_watch] if SHOW_TRAILS else None

time_text = ax.text(
    0.02, 0.95, '', transform=ax.transAxes,
    color='white', fontsize=11, fontweight='bold',
    verticalalignment='top', fontfamily='monospace'
)

# --- Animation update ---
def update(frame):
    # Dynamic order rate, based on get_order_rate and wall clock hour 
    order_rate = get_order_rate(sim)
    # if you need a harcoded value use 
    #order_rate = ORDER_RATE
    generate_orders(sim, rate_per_minute=order_rate)
    sim.run_tick()

    current_lons, current_lats = [], []
    for i, driver in enumerate(drivers_to_watch):
        if driver.coords != (0.0, 0.0):
            lon, lat = driver.coords[1], driver.coords[0]
        else:
            node_data = sub_graph.nodes[driver.location]
            lon, lat  = node_data['x'], node_data['y']

        current_lons.append(lon)
        current_lats.append(lat)

        if SHOW_TRAILS:
            trail_history[i].append((lon, lat))
            h_x, h_y = zip(*trail_history[i])
            lines[i].set_data(h_x, h_y)

    dots.set_offsets(np.c_[current_lons, current_lats])
    dots.set_color(colors)

    m = sim.metrics_snapshot()
    s = m['orders_by_status']
    hud = (
        f"t={int(sim.current_time)}s  [{m['dispatch_policy']} / {m['repositioning_policy']}]\n"
        f"Simulation time {sim.wall_clock_display}\n"
        f"PREP:{s['PREPARING']}  READY:{s['READY']}  "
        f"PICKUP:{s['PICKED_UP']}  DONE:{s['DELIVERED']}\n"
        f"idle drivers:{m['idle_drivers']}  "
        f"avg e2e:{int(m['avg_end_to_end_s'] or 0)}s"
    )
    time_text.set_text(hud)

    return ([dots, time_text] + lines) if SHOW_TRAILS else [dots, time_text]

# frames=X * step_size=Y => XY simulated seconds
ani = FuncAnimation(fig, update, frames=8640, interval=50, blit=True, repeat=False)
plt.close()
ani.save("sim_1d.mp4",writer='ffmpeg', fps=30)


In [10]:
# =========================================================
# POST-RUN METRICS
# Run after animation finishes or after sim.run_until(T).
# =========================================================


m = sim.metrics_snapshot()
print(f"Dispatch policy    : {m['dispatch_policy']}")
print(f"Reposition policy  : {m['repositioning_policy']}")
print(f"Simulated time     : {m['time']:.0f}s")
print(f"Total orders       : {m['total_orders']}")
print(f"Delivered          : {m['n_delivered']}")
print(f"Pending unassigned : {m['pending_unassigned']}")
print(f"Idle drivers       : {m['idle_drivers']}")
print()
print(f"Avg end-to-end     : {m['avg_end_to_end_s']:.1f}s"     if m['avg_end_to_end_s']     else "Avg end-to-end     : n/a")
print(f"Avg food wait      : {m['avg_food_wait_s']:.1f}s"      if m['avg_food_wait_s']      else "Avg food wait      : n/a")
print(f"Avg dispatch delay : {m['avg_dispatch_delay_s']:.1f}s"  if m['avg_dispatch_delay_s']  else "Avg dispatch delay : n/a")

# Per-order ledger — trace anything suspicious here
records = [
    {
        'order_id'         : o.id,
        'user_id'          : o.user_id,
        'driver_id'        : o.driver_id,
        'restaurant_id'    : o.restaurant_id,
        'start_time'       : o.start_time,
        'assigned_time'    : o.assigned_time,
        'pickup_time'      : o.pickup_time,
        'delivered_time'   : o.delivered_time,
        'end_to_end_s'     : o.end_to_end_time,
        'food_wait_s'      : o.food_wait_time,
        'dispatch_delay_s' : o.dispatch_delay,
        'status'           : o.status,
    }
    for o in sim.orders.values()
]

df = pd.DataFrame(records)



Dispatch policy    : HungarianPolicy
Reposition policy  : StaticPolicy
Simulated time     : 86430s
Total orders       : 10986
Delivered          : 10757
Pending unassigned : 129
Idle drivers       : 0

Avg end-to-end     : 1704.7s
Avg food wait      : 64.9s
Avg dispatch delay : 950.1s
